In [ ]:
## Import core components
from pyomo.environ import (
    Var,
    Param,
    Constraint,
    Expression,
    Objective,
    ConcreteModel,
    Block,
    value,
    assert_optimal_termination,
    units as pyunits,
)

# Ideas core components
from idaes.core import FlowsheetBlock
from idaes.core.util.scaling import (
    set_scaling_factor,
    constraint_scaling_transform,
)
from idaes.core.util.model_statistics import degrees_of_freedom

from pyomo.util.calc_var_value import calculate_variable_from_constraint

# Import reaktoro-pse and reaktoro
from reaktoro_pse.reaktoro_block import ReaktoroBlock
import reaktoro
from reaktoro_pse.core.util_classes.cyipopt_solver import (
    get_cyipopt_watertap_solver,
)


In [ ]:
sea_water_composition = {
    "Na": 10556,
    "K": 380,
    "Ca": 400,
    "Mg": 1262,
    "Cl": 18978,
    "SO4": 2649,
    "HCO3": 140,
}
# Get ions
ions = list(sea_water_composition.keys())
#

sea_water_ph = 7.56
mass_flow_dict={ion:value/1e6*pyunits.kg/pyunits.s for ion, value in sea_water_composition.items()}
mass_flow_dict['H2O']=1*pyunits.kg/pyunits.s
m = ConcreteModel()
# create IDAES flowsheet
m.fs = FlowsheetBlock(dynamic=False)
m.fs.mass_flow=Var(mass_flow_dict.keys(), initialize=lambda m, i: mass_flow_dict[i], units=pyunits.kg/pyunits.s)
m.fs.mass_flow.fix()
m.fs.pH=Var(initialize=sea_water_ph, units=pyunits.dimensionless)
m.fs.pH.fix()
m.fs.temperature=Var(initialize=293, units=pyunits.K)
m.fs.pressure=Var(initialize=101325, units=pyunits.Pa)
m.fs.H_modifier=Var(initialize=0, units=pyunits.mol/pyunits.s)
# can use NaCl or Sea water prop pack

In [ ]:
m.fs.reaktoro_outputs = Var(
        [("scalingTendency", "Calcite"), ("pH", None)],
        initialize=1,
    )
m.fs.eq_precipitation = ReaktoroBlock(
    aqueous_phase={
        "composition": m.fs.mass_flow,  # This is the speciesmass flow
        "convert_to_rkt_species": True,  # We can use default converter as its defined for default database (Phreeqc and pitzer)
        "activity_model": reaktoro.ActivityModelPitzer(),  # Can provide a string, or Reaktoro initialized class
        "fixed_solvent_specie": "H2O",  # We need to define our aqueous solvent as we have to speciate the block
    },
    system_state={
        "temperature": m.fs.temperature,
        "pressure": m.fs.pressure,
        "pH": m.fs.pH,
    },
    mineral_phase={"phase_components": ["Calcite"]},
    chemistry_modifier={
        "H": m.fs.H_modifier,
    },
    outputs=m.fs.reaktoro_outputs,  # This is the output dictionary we defined above
    database_file="pitzer.dat",  # needs to be a string that names the database file or points to its location
    build_speciation_block=True,  # This will build the constraints to speciate the block
    assert_charge_neutrality_on_property_block=False,
)

In [ ]:
import reaktoro as rkt

# lets get base configurations from our configured reaktoro block
reaktoro_block=m.fs.eq_precipitation

# get standard state we configured!
speciation_reaktoro_state=m.fs.eq_precipitation.speciation_block.rkt_state
print(speciation_reaktoro_state.state)



In [ ]:
# During reaktoro block initialization we first equalibrate the state 
# in the output above you can see that many species Are at 1e-16, they
# we created but have not be equalibrated yet, lets equlibarrte the state

rkt.equilibrate(speciation_reaktoro_state.state)
print(speciation_reaktoro_state.state)

In [ ]:
# As you can see its equlibrated, but not charged balanced, actually
# this is the most basic sepcitation calcualtion we can do, 
# there are not constraints to balance charge or even pH

# for example if we get aquous properties they look like this

print(rkt.AqueousProps(speciation_reaktoro_state.state))

# Do you see the pH is 7.73!!!!

In [ ]:
# In reaktoro-pse we must do something abit more complicated.
# We need to write ap roblem with constraintes where we set the pH, charge balance,
# and also enfore elemental balance (so we can change composition on the fly and get 
# derivatives as well)


# these objects are available in the reaktoro block, specifically, we will focus
# on reaktoro inputs, and grab equlibrium specs, which
# define all of our constraints and variables for our problem
# this defines our input specifications
rkt_input_spec= reaktoro_block.speciation_block.rkt_inputs
for key, ls in rkt_input_spec.constraint_dict.items():
    print(f"{key}: {ls}")
# this defines our equilibrium specification 
speciation_equilibrium_specs = rkt_input_spec.equilibrium_specs
print(speciation_equilibrium_specs)
# get existing variables 
existing_variables = speciation_equilibrium_specs.namesControlVariables()

# note the inputs for elements and H+ which is used to enforce our pH constraint

print(existing_variables)
existing_constraints = speciation_equilibrium_specs.namesConstraints()


# note how we have a constraint for charge, and each element except Cl As its used
# for charge balance (e.g. it is free) 
# also note we set the H and O dummy constraints, these are there to ensure we have as many constarints as variables, even though amouunt of O and H is 
# not fixed and only H2O is fixed. 
print(existing_constraints)

In [ ]:
# lets manualyl setup a solver and use our specs object to solve a reaktoro problem
solver = rkt.EquilibriumSolver(speciation_equilibrium_specs)
conditions = rkt.EquilibriumConditions(speciation_equilibrium_specs)

# lets edit solver options
solver_options = rkt.EquilibriumOptions()
solver_options.epsilon = 1e-32
solver_options.optima.maxiters=200
solver_options.optima.convergence.tolerance=1e-8
solver.setOptions(solver_options)

In [ ]:
# Lets configure our inputs!
conditions.set("pH", 7.56)
conditions.set("charge", 0)
conditions.temperature(293,'K')
conditions.pressure(101325,'Pa')
for ion, input_object in rkt_input_spec.rkt_inputs.items():
    if input_object.get_rkt_input_name() not in ['pH','temperature','pressure','charge']:
        print(f'Ion name based on RKT notation: {ion}, rkt name in equalbirium specs: {input_object.get_rkt_input_name()}')        
        mols=input_object.get_value(apply_conversion=True)
        conditions.set(input_object.get_rkt_input_name(), mols)
# Note - the rkt_input_object is tied to input variables for example
original=rkt_input_spec.rkt_inputs['Ca+2'].get_value(apply_conversion=True)
m.fs.mass_flow['Ca'].fix(m.fs.mass_flow['Ca']*2)
print('original Ca value:',original, 'updated Ca value:', rkt_input_spec.rkt_inputs['Ca+2'].get_value(apply_conversion=True))
# Lets solve our problem
result=solver.solve(speciation_reaktoro_state.state, conditions)
print('solve returned successful', result.succeeded())
assert result.succeeded()

In [ ]:
# check equilibrated state with our constraints
print(speciation_reaktoro_state.state)
print(rkt.AqueousProps(speciation_reaktoro_state.state))

# Note how the pH matches to what we set and our charge is now zero!

In [ ]:
# Lets now do our property block 
prop_reaktoro_state=reaktoro_block.rkt_state

prop_rkt_input_spec= reaktoro_block.rkt_inputs
for key, ls in prop_rkt_input_spec.constraint_dict.items():
    print(f"{key}: {ls}")

prop_equilibrium_specs = prop_rkt_input_spec.equilibrium_specs
print(prop_equilibrium_specs)
print(prop_equilibrium_specs.namesControlVariables())
print(prop_equilibrium_specs.namesConstraints())

# note how now we have constraints for all elements in system including charge - the charge is balanced by
# forcing H+/OH- balance

In [ ]:
# ocne again this state has not been updated in any way

In [ ]:
print(prop_reaktoro_state.state)

# what you will also note, no inputs have been actually updated
# its because we normally pass the true mol amounts from our property state into this block, lets do that  next

In [ ]:
for phase in prop_reaktoro_state.inputs.registered_phases:
    for species in prop_reaktoro_state.inputs.species_list[phase]:
        if species in prop_reaktoro_state.inputs:
            val= speciation_reaktoro_state.state.speciesAmount(species)
            prop_reaktoro_state.state.set(species, val,'mol')
            print(species, val)


In [ ]:
rkt.equilibrate(prop_reaktoro_state.state)
print(prop_reaktoro_state.state)

In [ ]:
# now lets set up our reaction configuration 
# lets manualyl setup a solver and use our specs object to solve a reaktoro problem
solver_props = rkt.EquilibriumSolver(prop_equilibrium_specs)
conditions_props = rkt.EquilibriumConditions(prop_equilibrium_specs)
solver_props.setOptions(solver_options)


In [ ]:
# Lets configure our inputs!
conditions_props.set("charge", 0)
conditions_props.temperature(293,'K')
conditions_props.pressure(101325,'Pa')
for ion, input_object in prop_rkt_input_spec.rkt_inputs.items():
    if input_object.get_rkt_input_name() not in ['pH','temperature','pressure','charge']:
        print(f'Ion name based on RKT notation: {ion}, rkt name in equalbirium specs: {input_object.get_rkt_input_name()}')        
        if 'modifier' in input_object.get_rkt_input_name():
            print('skipping modifier')
            continue
        # mols=input_object.get_value(apply_conversion=True)
        mols= speciation_reaktoro_state.state.speciesAmount(ion) # grab this from our speciation state which is now updated with our input values
        conditions_props.set(input_object.get_rkt_input_name(), mols)
H_modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(H_modifier_name, 1e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem
result=solver_props.solve(prop_reaktoro_state.state, conditions_props)
print('solve returned successful', result.succeeded())
assert result.succeeded()

In [ ]:
# review the state
print(prop_reaktoro_state.state)
print(rkt.AqueousProps(prop_reaktoro_state.state))

In [ ]:
modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(modifier_name, 1000e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem

solver_options.optima.convergence.tolerance=1e-8
solver_options.optima.output.active = True
solver_props.setOptions(solver_options)
result=solver_props.solve(prop_reaktoro_state.state, conditions_props)

print('solve returned successful', result.succeeded())
print(prop_reaktoro_state.state)

print(rkt.AqueousProps(prop_reaktoro_state.state))

In [ ]:
modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(modifier_name, 10e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem
solver_options.optima.output.active = True

solver_options.optima.backtracksearch.apply_min_max_fix_and_accept = True
solver_options.optima.convergence.tolerance=1e-8
solver_options.logarithm_barrier_factor=100
solver_options.epsilon = 1e-16
solver_options.optima.linesearch.tolerance =1e-5
solver_options.optima.linesearch.trigger_when_current_error_is_greater_than_initial_error_by_factor =1
solver_options.optima.linesearch.trigger_when_current_error_is_greater_than_previous_error_by_factor  =2
solver_options.optima.steepestdescent.tolerance = 1e-85
solver_options.optima.steepestdescent.maxiters =  20
solver_options.optima.maxiters=50
solver_props.setOptions(solver_options)

result=solver_props.solve(prop_reaktoro_state.state, conditions_props)

print('solve returned successful', result.succeeded())
print(prop_reaktoro_state.state)

print(rkt.AqueousProps(prop_reaktoro_state.state))

In [ ]:
modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(modifier_name, 1000e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem
solver_options.optima.output.active = True
solver_options.optima.convergence.tolerance=1e-8
solver_props.setOptions(solver_options)
result=solver_props.solve(prop_reaktoro_state.state, conditions_props)

print('solve returned successful', result.succeeded())
print(prop_reaktoro_state.state)
print(rkt.AqueousProps(prop_reaktoro_state.state))